In [1]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models  # Keras 레이어 / 모델 유틸

# 데이터셋 로드 및 전처리
(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()

# 픽셀 값 정규화
train_images = tf.image.resize(train_images, (224, 224)) / 255.0  # MobileNet 입력크기 224x224로 리사이즈 후 정규화
test_images =  tf.image.resize(test_images, (224, 224)) / 255.0

# 사전학습된 MobileNet 모델 로드 (ImageNet 학습 가중치 사용, 분류기 제거)
base_model = tf.keras.applications.MobileNet(weights = 'imagenet', include_top = False, input_shape=(224, 224, 3))

base_model.trainable = False  # 가중치 고정

inputs = layers.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)  # base 모델 통과
x = layers.GlobalAveragePooling2D()(x)  # 공간 차원 평균으로 벡터화(평탄화)
outputs = layers.Dense(10, activation='softmax')(x)  # CIFAR-10용 10클래스 분기
model = models.Model(inputs, outputs)  # 최종 전이학습 모델

model.summary()

c:\Users\Playdata\multimodal\multi_venv\Lib\site-packages\keras\src\datasets\cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenet_1.00_224 (Functional) │ (None, 7, 7, 1024)     │     3,228,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        10,250 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,239,114 (12.36 MB)

 Trainable params: 10,250 (40.04 KB)

 Non-trainable params: 3,228,864 (12.32 MB)

In [2]:
# 모델 컴파일
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 모델 학습 (10% 검증데이터)
model.fit(train_images, train_labels, epochs=5, batch_size=32, validation_split=0.1)

Epoch 1/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 395s 279ms/step - accuracy: 0.7979 - loss: 0.5927 - val_accuracy: 0.8502 - val_loss: 0.4195
Epoch 2/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 687s 489ms/step - accuracy: 0.8619 - loss: 0.4004 - val_accuracy: 0.8624 - val_loss: 0.3907
Epoch 3/5
 371/1407 ━━━━━━━━━━━━━━━━━━━━ 4:35 266ms/step - accuracy: 0.8779 - loss: 0.3512

KeyboardInterrupt: 

In [ ]:
# 모델 평가
test_loss, test_acc = model.evaluate(test_images, test_labels)
print('테스트 정확도:', test_acc)

In [ ]:
model.save('mobilenet_cifar10.h5')  # 모델 저장

In [ ]:
import os

print('모델 파일 크기:', os.path.getsize('mobilenet_cifar10.h5') / 1e6, 'MB')  # 바이트 -> MB